# Foundations Data Tour And SQL Features

This notebook is the shared starting path for the v0.1 curriculum. It uses tiny deterministic data to inspect the **Realistic synthetic data model**, work with **Progressive data views**, load tables into SQLite, extract SQL features, trace the **Alert lifecycle**, and evaluate simple alert scores without using protected answer keys.

In [ ]:
from pathlib import Path

import pandas as pd

from banking_fraud_lab import (
    FOUNDATION_PROGRESSIVE_VIEW_SPECS,
    build_learner_facing_views,
    build_foundation_progressive_views,
    create_minimal_banking_world_sqlite,
    evaluate_alert_scores,
    generate_minimal_banking_world,
)
from banking_fraud_lab.schema import PROTECTED_SCENARIO_ANSWER_KEYS

## Generate Tiny Data

The generated tables form one consistent banking world rather than disconnected toy tables. The default learner path uses **Progressive data views**: learners can inspect Clients, Users, Banking relationships, transactions, alerts, cases, and outcomes while protected scenario answer keys stay outside the normal view.

In [ ]:
tables = generate_minimal_banking_world(seed=42)
learner_tables = build_learner_facing_views(tables)

assert PROTECTED_SCENARIO_ANSWER_KEYS in tables
assert PROTECTED_SCENARIO_ANSWER_KEYS not in learner_tables

table_counts = pd.DataFrame(
    [
        {"table_name": table_name, "row_count": len(frame)}
        for table_name, frame in learner_tables.items()
    ]
)
table_counts

## Inspect Foundation Progressive Views

Foundation Progressive data views are smaller query surfaces built from the canonical model. They introduce common joins while preserving traceability back to Clients, Partners, Banking relationships, suspicious activities, alerts, cases, and outcomes.

In [ ]:
foundation_views = build_foundation_progressive_views(tables)
foundation_view_contract = pd.DataFrame(
    [
        {
            "view_name": spec.name,
            "source_tables": ", ".join(spec.source_tables),
            "column_count": len(spec.columns),
            "stable_for_case_references": spec.stable_for_case_references,
        }
        for spec in FOUNDATION_PROGRESSIVE_VIEW_SPECS
    ]
)

assert set(foundation_views) == {spec.name for spec in FOUNDATION_PROGRESSIVE_VIEW_SPECS}
assert PROTECTED_SCENARIO_ANSWER_KEYS not in foundation_views
assert set(foundation_views["foundation_client_relationships"]["client_id"]) <= set(
    tables["clients"]["client_id"]
)
foundation_view_contract

## Load SQLite

SQLite is the first SQL surface for the curriculum. The loader creates tables from the schema contract, keeps foreign keys enabled, excludes protected answer keys by default, and exposes the foundation Progressive data views as queryable SQLite views.

In [ ]:
connection = create_minimal_banking_world_sqlite(":memory:", seed=42)

sqlite_tables = pd.read_sql_query(
    """
    SELECT name AS table_name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name
    """,
    connection,
)
assert PROTECTED_SCENARIO_ANSWER_KEYS not in set(sqlite_tables["table_name"])

sqlite_views = pd.read_sql_query(
    """
    SELECT name AS view_name
    FROM sqlite_master
    WHERE type = 'view'
    ORDER BY name
    """,
    connection,
)
assert set(sqlite_views["view_name"]) == {
    spec.name for spec in FOUNDATION_PROGRESSIVE_VIEW_SPECS
}
sqlite_views

## Extract SQL Features

The first feature query joins account and **Banking relationship** context to transactions and uses a window function to count transaction activity by relationship.

In [ ]:
transaction_features = pd.read_sql_query(
    """
    SELECT
        t.transaction_id,
        a.institution_name,
        br.banking_relationship_id,
        br.relationship_manager_code,
        CAST(t.amount_chf AS REAL) AS amount_chf,
        COUNT(*) OVER (
            PARTITION BY br.banking_relationship_id
        ) AS relationship_transaction_count,
        CASE WHEN CAST(t.amount_chf AS REAL) >= 50000 THEN 1 ELSE 0 END
            AS high_value_movement
    FROM transactions AS t
    JOIN accounts AS a
        ON t.account_id = a.account_id
    JOIN banking_relationships AS br
        ON a.banking_relationship_id = br.banking_relationship_id
    ORDER BY t.booked_at
    """,
    connection,
)
assert len(transaction_features) == len(learner_tables["transactions"])
assert transaction_features["relationship_transaction_count"].max() >= 1
transaction_features.head()

## Run Foundation SQL Exercises

The SQL exercise files extend the first feature query into joins, windows, cohorts, alert queues, and feature extraction. They use canonical tables together with foundation Progressive data views.

In [ ]:
current_path = Path.cwd()
repo_root = current_path if (current_path / "sql").exists() else current_path.parents[1]
expanded_sql_exercise_paths = [
    repo_root / "sql/examples/01_alert_lifecycle_join.sql",
    repo_root / "sql/examples/02_alert_review_window.sql",
    repo_root / "sql/examples/03_client_relationship_cohorts.sql",
    repo_root / "sql/examples/04_progressive_alert_queue.sql",
    repo_root / "sql/examples/05_transaction_feature_extraction.sql",
]

expanded_sql_exercise_summary = []
for exercise_path in expanded_sql_exercise_paths:
    exercise_result = pd.read_sql_query(
        exercise_path.read_text(encoding="utf-8"),
        connection,
    )
    assert not exercise_result.empty
    expanded_sql_exercise_summary.append(
        {
            "exercise": exercise_path.name,
            "row_count": len(exercise_result),
            "column_count": len(exercise_result.columns),
        }
    )

pd.DataFrame(expanded_sql_exercise_summary)

## Trace The Alert Lifecycle

The **Alert lifecycle** starts with suspicious activity, then moves through alerts, cases, and case outcomes. Confirmed fraud is an investigation outcome, not a single transaction flag.

In [ ]:
alert_lifecycle = pd.read_sql_query(
    """
    SELECT
        sa.suspicious_activity_id,
        sa.activity_type,
        al.alert_id,
        al.alert_status,
        c.case_id,
        co.outcome_type,
        co.confirmed_fraud
    FROM suspicious_activities AS sa
    LEFT JOIN alerts AS al
        ON sa.suspicious_activity_id = al.suspicious_activity_id
    LEFT JOIN cases AS c
        ON al.alert_id = c.alert_id
    LEFT JOIN case_outcomes AS co
        ON c.case_id = co.case_id
    ORDER BY sa.detected_at
    """,
    connection,
)
assert alert_lifecycle["alert_id"].notna().all()
assert alert_lifecycle["case_id"].notna().sum() >= 1
alert_lifecycle

## Evaluate Alert Scores

A simple score can be evaluated with alert-aware metrics. The report focuses on precision, recall, PR-AUC, alert capacity, threshold tradeoffs, and costs. Headline accuracy is left out because fraud outcomes are sparse and operationally mediated.

In [ ]:
labeled_cases = pd.read_sql_query(
    """
    SELECT
        c.case_id,
        c.alert_id,
        al.alert_type,
        co.confirmed_fraud,
        co.loss_amount_chf
    FROM cases AS c
    JOIN alerts AS al
        ON c.alert_id = al.alert_id
    JOIN case_outcomes AS co
        ON c.case_id = co.case_id
    ORDER BY c.opened_at
    """,
    connection,
)
alert_scores = pd.DataFrame(
    {
        "alert_id": labeled_cases["alert_id"],
        "score": labeled_cases["alert_type"].map(
            {"new_beneficiary_payment": 0.9, "session_payment_velocity": 0.35}
        ),
    }
)

report = evaluate_alert_scores(
    labeled_cases[["case_id", "alert_id"]],
    labeled_cases[["case_id", "confirmed_fraud", "loss_amount_chf"]],
    alert_scores,
    thresholds=(0.3, 0.5, 0.8),
    alert_capacity=1,
    investigation_cost_chf=50.0,
    false_positive_cost_chf=10.0,
)

threshold_summary = pd.DataFrame(report["threshold_summaries"])
threshold_summary.insert(0, "pr_auc", report["pr_auc"])
assert report["population"]["case_count"] == len(labeled_cases)
assert "accuracy is out of scope" in report["limitation_summary"]
threshold_summary[
    [
        "pr_auc",
        "threshold",
        "precision",
        "recall",
        "alert_volume",
        "alert_capacity",
        "over_capacity_alerts",
        "total_cost_chf",
    ]
]

In [ ]:
connection.close()